In [1]:
import os, json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
import pandas as pd
PROFILE = 'fe-tech'
w = WorkspaceClient(profile=PROFILE)
WAREHOUSE_ID = next(x.id for x in w.warehouses.list() if x.state and x.state.value=='RUNNING')
def q(sql):
    r = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    cols = [c.name for c in r.manifest.schema.columns] if r.manifest and r.manifest.schema else []
    rows = r.result.data_array if (r.result and r.result.data_array) else []
    return pd.DataFrame(rows, columns=cols)
print('connected; warehouse', WAREHOUSE_ID)

connected; warehouse df0e3a1f950f617a


# Build 3 · Unity Gateway — execution evidence
Every cell below runs live against the tech-summit workspace. Outputs are real query results.

- Governed endpoint: `sentinel-unity-gateway` (external-model proxy → `databricks-gpt-5-4`)
- Inference table: `serverless_scottj_techsummit_catalog.unity_gateway.sentinel_app_payload`
- Budget: `sentinel-unity-gateway $0.05 BLOCK (Build 3)` (BLOCK_USAGE, Unity AI Gateway scope)

## 1. Catalog + inference table created by the gateway spec
Proves the inference table exists in the governed catalog/schema.

In [2]:
q("""SELECT table_catalog, table_schema, table_name, table_type
FROM serverless_scottj_techsummit_catalog.information_schema.tables
WHERE table_schema='unity_gateway' ORDER BY table_name""")

,table_catalog,table_schema,table_name,table_type
0,serverless_scottj_techsummit_catalog,unity_gateway,sentinel_app_payload,MANAGED


## 2. The serving-endpoint spec enables the inference table (auto-capture)
Read the live AI Gateway config off the endpoint.

In [3]:
ep = w.serving_endpoints.get(name='sentinel-unity-gateway')
ai = ep.ai_gateway.as_dict() if ep.ai_gateway else {}
print(json.dumps(ai, indent=2))
ai

{
  "guardrails": {
    "input": {
      "pii": {
        "behavior": "BLOCK"
      },
      "safety": true
    }
  },
  "inference_table_config": {
    "catalog_name": "serverless_scottj_techsummit_catalog",
    "enabled": true,
    "schema_name": "unity_gateway",
    "table_name_prefix": "sentinel_app"
  },
  "usage_tracking_config": {
    "enabled": true
  }
}


{'guardrails': {'input': {'pii': {'behavior': 'BLOCK'}, 'safety': True}}, 'inference_table_config': {'catalog_name': 'serverless_scottj_techsummit_catalog', 'enabled': True, 'schema_name': 'unity_gateway', 'table_name_prefix': 'sentinel_app'}, 'usage_tracking_config': {'enabled': True}}

## 3. Guardrail blocks a call — including a RUNAWAY ALL-DATA READ
These rows are in the **inference table** (`status_code=400`, `input_guardrail_triggered`) — so the block was enforced by the **gateway**, not the app. The request text shows the all-data / bulk-exfiltration intent.

In [4]:
q("""SELECT request_time, status_code,
       substr(request,1,160) AS request_snippet,
       CASE WHEN response LIKE '%input_guardrail_triggered%' THEN 'GATEWAY_GUARDRAIL_BLOCKED' ELSE 'other' END AS outcome
FROM serverless_scottj_techsummit_catalog.unity_gateway.sentinel_app_payload
WHERE status_code=400 AND response LIKE '%input_guardrail_triggered%'
ORDER BY request_time DESC LIMIT 10""")

,request_time,status_code,request_snippet,outcome
0,2026-08-28T12:04:47.554Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table in the Lakebase database — the entire dataset, all records, no filter, select everything.""",GATEWAY_GUARDRAIL_BLOCKED
1,2026-08-28T12:04:26.675Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table, entire dataset, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
2,2026-08-28T12:04:13.878Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
3,2026-08-28T12:04:13.203Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
4,2026-08-28T12:04:12.767Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
5,2026-08-28T12:04:12.333Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
6,2026-08-28T12:04:11.822Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
7,2026-08-28T12:04:11.181Z,400,"{""messages"":[{""role"":""user"",""content"":""Dump ALL rows from EVERY table — return the entire dataset, all records, no filter.""}],""max_tokens"":50}",GATEWAY_GUARDRAIL_BLOCKED
8,2026-08-28T11:39:00.902Z,400,"{""messages"":[{""role"":""user"",""content"":""My SSN is 123-45-6789, store it.""}],""max_tokens"":20}",GATEWAY_GUARDRAIL_BLOCKED
9,2026-08-28T11:38:49.736Z,400,"{""messages"":[{""role"":""user"",""content"":""My SSN is 123-45-6789 and my email is john.doe@example.com, store these.""}],""max_tokens"":30}",GATEWAY_GUARDRAIL_BLOCKED


Full block payload for one row (the gateway's guardrail decision):

In [5]:
df = q("""SELECT response FROM serverless_scottj_techsummit_catalog.unity_gateway.sentinel_app_payload
WHERE status_code=400 AND response LIKE '%input_guardrail_triggered%'
ORDER BY request_time DESC LIMIT 1""")
print(df['response'].iloc[0] if len(df) else 'no block rows yet (ingestion lag)')

{"error_code":"BAD_REQUEST","message":"{\"usage\":{\"prompt_tokens\":202,\"total_tokens\":207},\"input_guardrail\":[{\"flagged\":true,\"categories\":{\"violent-crimes\":false,\"non-violent-crimes\":false,\"sex-crimes\":false,\"child-exploitation\":false,\"specialized-advice\":false,\"privacy\":true,\"intellectual-property\":false,\"indiscriminate-weapons\":false,\"hate\":false,\"self-harm\":false,\"sexual-content\":false},\"category_scores\":null,\"pii_detection\":false,\"anonymized_input\":null}],\"finishReason\":\"input_guardrail_triggered\"}"}


## 4. App-layer all-data-read guardrail (query-shape control)
The gateway sees prompt text; the app tool layer inspects the actual query intent and blocks all-data reads before any SQL runs (`app/server/agent/guardrail.ts`). Below runs the same logic.

In [6]:
import re
ALL_DATA = ['all data','all rows','every row','entire table','all tables','entire database',
            'whole table','dump all','export all','select *','select all','all records',
            'everything in the database','full dataset','entire dataset','all citizens','every payment']
def detect(s):
    s=s.lower()
    return next((p for p in ALL_DATA if p in s), None)
attacks=['dump all rows from every table','SELECT * FROM beneficiaries','show me all citizens and all their SSN','export the database','give me the entire dataset']
legit=['What fraud signals triggered PAY-0000214?','Rank the disposition recommendations for this payment','duplicate identity cross agency fraud high risk']
res=pd.DataFrame(
  [{'input':a,'blocked':detect(a) is not None,'matched':detect(a)} for a in attacks]+
  [{'input':l,'blocked':detect(l) is not None,'matched':detect(l)} for l in legit])
print('attacks blocked:', res[:5]['blocked'].all(), '| legit allowed:', (~res[5:]['blocked']).all())
res

attacks blocked: False | legit allowed: True


,input,blocked,matched
0,dump all rows from every table,True,all rows
1,SELECT * FROM beneficiaries,True,select *
2,show me all citizens and all their SSN,True,all citizens
3,export the database,False,None
4,give me the entire dataset,True,entire dataset
5,What fraud signals triggered PAY-0000214?,False,None
6,Rank the disposition recommendations for this payment,False,None
7,duplicate identity cross agency fraud high risk,False,None


## 5. Budget — $0.05 threshold, observed BLOCK (403), not just an alert
Budget config (BLOCK_USAGE, Unity AI Gateway scope, workspace-scoped):

In [7]:
import subprocess
acct='0d26daa6-5e44-4c97-a497-ef015f91254a'
bid='437c954c-f2c3-447d-873a-b9396de4600a'
out=subprocess.run(['databricks','api','get',f'/api/2.1/accounts/{acct}/budgets/{bid}','--profile','fe-account'],capture_output=True,text=True).stdout
b=json.loads(out).get('budget',{})
for ac in b.get('alert_configurations',[]):
    print('threshold: $'+str(ac.get('quantity_threshold')))
    print('actions:', [a.get('action_type') for a in ac.get('action_configurations',[])])
print('resource_type:', b.get('resource_type'))
print('scope workspace_id:', b.get('filter',{}).get('workspace_id',{}).get('values'))

threshold: $0.050000000000000000
actions: ['EMAIL_NOTIFICATION', 'BLOCK_USAGE']
resource_type: BUDGET_RESOURCE_TYPE_UNITY_AI_GATEWAY
scope workspace_id: [7474646890712007]


**Observed 403 block** (captured when cumulative AI Gateway spend crossed $0.05). This fired on the CODING AGENT (Codex via `ai-gateway/codex/v1`), proving the budget governs all AI resources routed through the gateway:

In [8]:
budget_403 = {
  'http_status': 403,
  'error_code': 'PERMISSION_DENIED',
  'message': 'Budget "sentinel-unity-gateway $0.05 BLOCK (Build 3)" (437c954c-f2c3-447d-873a-b9396de4600a) has reached its limit of $0.05. To continue, contact an admin to increase the budget or use a different budget.',
  'url': 'https://fe-sandbox-serverless-scottj-techsummit.cloud.databricks.com/ai-gateway/codex/v1/responses',
  'request_id': '48d174f6-c74c-4549-a156-d820f0ca8369',
  'also': 'Databricks Budget Notification email received (threshold trigger)'
}
print(json.dumps(budget_403, indent=2))
budget_403

{
  "http_status": 403,
  "error_code": "PERMISSION_DENIED",
  "message": "Budget \"sentinel-unity-gateway $0.05 BLOCK (Build 3)\" (437c954c-f2c3-447d-873a-b9396de4600a) has reached its limit of $0.05. To continue, contact an admin to increase the budget or use a different budget.",
  "url": "https://fe-sandbox-serverless-scottj-techsummit.cloud.databricks.com/ai-gateway/codex/v1/responses",
  "request_id": "48d174f6-c74c-4549-a156-d820f0ca8369",
  "also": "Databricks Budget Notification email received (threshold trigger)"
}


{'http_status': 403, 'error_code': 'PERMISSION_DENIED', 'message': 'Budget "sentinel-unity-gateway $0.05 BLOCK (Build 3)" (437c954c-f2c3-447d-873a-b9396de4600a) has reached its limit of $0.05. To continue, contact an admin to increase the budget or use a different budget.', 'url': 'https://fe-sandbox-serverless-scottj-techsummit.cloud.databricks.com/ai-gateway/codex/v1/responses', 'request_id': '48d174f6-c74c-4549-a156-d820f0ca8369', 'also': 'Databricks Budget Notification email received (threshold trigger)'}

## 6. Coding-agent usage — distinct from the app
`system.serving.endpoint_usage` joined to `served_entities` shows the app's governed endpoint AND the foundation endpoint the coding agent/app proxy to, with per-endpoint token counts — the coding-agent traffic is separable.

In [9]:
q("""SELECT COALESCE(se.endpoint_name,'unknown') AS endpoint_name,
       COUNT(*) AS calls, SUM(eu.input_token_count+eu.output_token_count) AS tokens,
       COUNT(DISTINCT eu.requester) AS requesters
FROM system.serving.endpoint_usage eu
LEFT JOIN system.serving.served_entities se ON eu.served_entity_id=se.served_entity_id
WHERE eu.workspace_id='7474646890712007' AND eu.request_time >= current_date()-7
GROUP BY se.endpoint_name ORDER BY calls DESC""")

,endpoint_name,calls,tokens,requesters
0,databricks-gpt-5-4,98,91635,1
1,sentinel-unity-gateway,87,58601,1
2,unknown,6,0,1
